# PDF RAG with LangChain

This notebook builds your retrieval index in one folder: **`rag_index/`**

| Step | What happens | Output |
|------|----------------|--------|
| 1 | Setup paths | folders created |
| 2 | List PDFs | console list |
| 3 | **Parse + chunk** (Docling) | `parsed_markdown/*.md`, `chunks.json` |
| 4 | **Embed** (BGE) | `faiss.index`, `metadata.json` |
| 5 | Test search | top chunks printed |
| 6 | Test answer (Qwen via Ollama) | sample answer |

After this, run Streamlit:
```bash
streamlit run streamlit_app.py
```

## About console messages you may see

**`HF_TOKEN` / unauthenticated Hugging Face** — Optional. The embedding model downloads from Hugging Face. It still works without a token; setting `export HF_TOKEN=your_token` gives higher rate limits.

**`RapidOCR returned empty result`** — Comes from Docling’s OCR on some PDF pages (blank slides, images with no text, or pages that already have embedded text). Often **harmless**: Docling still reads the PDF. If chunk counts look reasonable at the end, you can ignore these.

**`Loading weights`** — Docling / layout models loading on GPU — normal on first run.

**`Token indices sequence length ... > 512`** — Docling’s chunker tokenizer warning while splitting long sections. Usually safe; chunks are still created.

## WSL / VS Code disconnecting during Step 3?

Docling is **very RAM-heavy** (layout models + OCR). Large PDFs can exceed 15GB and Linux **OOM-kills** the process → VS Code shows "Reconnecting...".

**Fixes:**
1. In Windows `C:\Users\<you>\.wslconfig` set `memory=24GB` and `swap=8GB`, then `wsl --shutdown`
2. In Step 1, keep `DO_OCR = False` for normal text PDFs (saves a lot of RAM)
3. **Run Step 3 from a plain WSL terminal** (not VS Code): `jupyter notebook` — resume still works via `chunks.json`
4. Re-run Step 3 anytime — it skips PDFs already done
5. **"Kernel is dead"** on `Python.pdf` was a **crash (segfault)** in Docling's default PDF parser — not lack of RAM. Step 1 now uses Docling's **PyPdfium backend** instead (still Docling, still HybridChunker).

## Step 1 — Setup

In [1]:
from pathlib import Path
import gc
import json
import logging
import os
import warnings

import faiss
import numpy as np
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.backend.pypdfium2_backend import PyPdfiumDocumentBackend
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling_core.transforms.chunker.hybrid_chunker import HybridChunker
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from ollama import chat

# Quieter logs during parsing (OCR empty-page noise is common and usually OK)
logging.getLogger("RapidOCR").setLevel(logging.CRITICAL)
logging.getLogger("docling").setLevel(logging.WARNING)
logging.getLogger("transformers").setLevel(logging.ERROR)
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")
warnings.filterwarnings("ignore", message=".*unauthenticated.*HF Hub.*")
warnings.filterwarnings("ignore", message=".*sequence length is longer than.*")


def page_from_meta(meta) -> int | None:
    """Read page number from Docling DocMeta (Pydantic), not a dict."""
    if meta is None:
        return None
    if isinstance(meta, dict):
        return meta.get("page_no") or meta.get("page_number") or meta.get("page")
    for item in getattr(meta, "doc_items", None) or []:
        for prov in getattr(item, "prov", None) or []:
            page_no = getattr(prov, "page_no", None)
            if page_no is not None:
                return int(page_no)
    return None


def section_from_meta(meta) -> str | None:
    headings = getattr(meta, "headings", None) if meta is not None else None
    if headings:
        return " > ".join(headings)
    return None

ROOT = Path(".").resolve()
if not (ROOT / "knowledge-base").exists():
    ROOT = Path("RAG With LangChain").resolve()

RAG_INDEX = ROOT / "rag_index"
PARSED_MD = RAG_INDEX / "parsed_markdown"
CHUNKS_PATH = RAG_INDEX / "chunks.json"
INDEX_PATH = RAG_INDEX / "faiss.index"
METADATA_PATH = RAG_INDEX / "metadata.json"
KB = ROOT / "knowledge-base"
EMBEDDING_MODEL = "BAAI/bge-base-en-v1.5"
OLLAMA_MODEL = "qwen3:8b"

# --- Docling memory tuning (important on WSL) ---
# False = much less RAM if your PDFs have normal embedded text (not scanned images)
DO_OCR = False
DO_TABLE_STRUCTURE = False
FORCE_BACKEND_TEXT = True


def make_converter() -> DocumentConverter:
  """Fresh Docling converter per PDF. Uses PyPdfium PDF backend — the default
  docling-parse backend segfaults on some PDFs (e.g. Python.pdf) in WSL."""
  opts = PdfPipelineOptions(
      do_ocr=DO_OCR,
      do_table_structure=DO_TABLE_STRUCTURE,
      force_backend_text=FORCE_BACKEND_TEXT,
      generate_page_images=False,
      generate_picture_images=False,
  )
  return DocumentConverter(
      format_options={
          InputFormat.PDF: PdfFormatOption(
              pipeline_options=opts,
              backend=PyPdfiumDocumentBackend,
          )
      }
  )


def print_ram(label: str) -> None:
  try:
      import psutil

      mb = psutil.Process().memory_info().rss / (1024 * 1024)
      print(f"      → RAM: {mb:.0f} MB ({label})")
  except ImportError:
      pass


def free_memory() -> None:
  gc.collect()


RAG_INDEX.mkdir(parents=True, exist_ok=True)
PARSED_MD.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT)
print("Knowledge base:", KB)
print("Index output:", RAG_INDEX)
print("Embedding model:", EMBEDDING_MODEL)
print("Chat model (Ollama):", OLLAMA_MODEL)
print(f"Docling: OCR={DO_OCR}, tables={DO_TABLE_STRUCTURE}, backend_text={FORCE_BACKEND_TEXT}")

/home/abiram/Prayoga/Marga/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project root: /home/abiram/Prayoga/Marga/RAG With LangChain
Knowledge base: /home/abiram/Prayoga/Marga/RAG With LangChain/knowledge-base
Index output: /home/abiram/Prayoga/Marga/RAG With LangChain/rag_index
Embedding model: BAAI/bge-base-en-v1.5
Chat model (Ollama): qwen3:8b
Docling: OCR=False, tables=False, backend_text=True


## Step 2 — List PDFs in `knowledge-base/`

In [2]:
import fitz

pdf_files = sorted(KB.glob("*.pdf"))
if not pdf_files:
    raise FileNotFoundError(f"No PDF files found in {KB}")

print(f"Found {len(pdf_files)} PDF(s):\n")
for i, pdf_path in enumerate(pdf_files, start=1):
    size_mb = pdf_path.stat().st_size / (1024 * 1024)
    with fitz.open(pdf_path) as doc:
        pages = doc.page_count
    print(f"  {i}. {pdf_path.name}  ({size_mb:.2f} MB, {pages} pages)")

Found 4 PDF(s):

  1. AI Engineering.pdf  (31.92 MB, 535 pages)
  2. Data Science.pdf  (88.26 MB, 531 pages)
  3. ML.pdf  (5.44 MB, 118 pages)
  4. Python.pdf  (65.63 MB, 68 pages)


## Step 3 — Parse PDFs with Docling and chunk

**Output folder:** `rag_index/`
- Markdown: `rag_index/parsed_markdown/<pdf-name>.md`
- Chunks JSON: `rag_index/chunks.json` (saved after **each** PDF — safe to resume)

**Resume rules (re-run this cell anytime):**
1. PDF already in `chunks.json` → **skip entirely**
2. Markdown `.md` exists but no chunks yet → **skip Docling parse**, chunk from markdown
3. No markdown → **full Docling parse** + HybridChunker chunking

This step can take several minutes per large PDF (layout + optional OCR).

In [3]:
import threading
import time

from langchain_text_splitters import RecursiveCharacterTextSplitter

chunker = HybridChunker()


def run_docling_parse(pdf_path: Path):
    """Parse one PDF with a heartbeat — Docling prints nothing while working."""
    with fitz.open(pdf_path) as doc:
        page_count = doc.page_count
    print(f"      → {page_count} pages — layout model runs on each (no live progress bar)")

    done = threading.Event()
    start = time.time()

    def heartbeat() -> None:
        while not done.wait(30):
            elapsed = int(time.time() - start)
            print(
                f"      … still parsing ({elapsed // 60}m {elapsed % 60:02d}s elapsed)"
            )

    threading.Thread(target=heartbeat, daemon=True).start()
    converter = make_converter()
    try:
        result = converter.convert(pdf_path)
        elapsed = int(time.time() - start)
        print(f"      → Docling finished in {elapsed // 60}m {elapsed % 60:02d}s")
        return result
    finally:
        done.set()
md_splitter = RecursiveCharacterTextSplitter(chunk_size=1200, chunk_overlap=200)


def normalize(text: str) -> str:
    return " ".join(text.split()).strip()


def md_path_for(pdf_path: Path) -> Path:
    return PARSED_MD / f"{pdf_path.stem}.md"


def load_saved_chunks() -> list[Document]:
    if not CHUNKS_PATH.exists():
        return []
    rows = json.loads(CHUNKS_PATH.read_text(encoding="utf-8"))
    return [
        Document(page_content=row["text"], metadata=row.get("metadata", {}))
        for row in rows
    ]


def save_all_chunks(documents: list[Document]) -> None:
    payload = [{"text": d.page_content, "metadata": d.metadata} for d in documents]
    CHUNKS_PATH.write_text(json.dumps(payload, indent=2), encoding="utf-8")


def chunk_with_hybrid(dl_doc, pdf_path: Path) -> list[Document]:
    out: list[Document] = []
    for i, chunk in enumerate(chunker.chunk(dl_doc=dl_doc), start=1):
        text = normalize(getattr(chunk, "text", "") or "")
        if not text:
            continue
        meta = getattr(chunk, "meta", None)
        out.append(
            Document(
                page_content=text,
                metadata={
                    "source": pdf_path.name,
                    "page_number": page_from_meta(meta),
                    "section": section_from_meta(meta),
                    "chunk_index": i,
                    "chunk_type": "docling_hybrid",
                },
            )
        )
    return out


def chunk_from_markdown(md_path: Path, pdf_path: Path) -> list[Document]:
    text = md_path.read_text(encoding="utf-8")
    out: list[Document] = []
    for i, piece in enumerate(md_splitter.split_text(text), start=1):
        piece = normalize(piece)
        if not piece:
            continue
        out.append(
            Document(
                page_content=piece,
                metadata={
                    "source": pdf_path.name,
                    "page_number": None,
                    "section": None,
                    "chunk_index": i,
                    "chunk_type": "markdown_resume",
                },
            )
        )
    return out


documents = load_saved_chunks()
chunked_sources = {d.metadata.get("source") for d in documents}
parse_report: list[dict] = []
total_pdfs = len(pdf_files)

print("=" * 60)
print("PARSING & CHUNKING")
print(f"Chunks file: {CHUNKS_PATH}")
print("=" * 60)
print("\nResume check:")
for pdf_path in pdf_files:
    md_path = md_path_for(pdf_path)
    in_chunks = pdf_path.name in chunked_sources
    print(
        f"  • {pdf_path.name}: markdown={'yes' if md_path.exists() else 'no'}"
        f" | chunks={'yes' if in_chunks else 'no'}"
    )

for pdf_no, pdf_path in enumerate(pdf_files, start=1):
    md_path = md_path_for(pdf_path)

    if pdf_path.name in chunked_sources:
        n = sum(1 for d in documents if d.metadata.get("source") == pdf_path.name)
        print(f"\n[{pdf_no}/{total_pdfs}] SKIP {pdf_path.name} — already in chunks.json ({n} chunks)")
        parse_report.append(
            {
                "pdf": pdf_path.name,
                "status": "skipped",
                "markdown_file": md_path.name if md_path.exists() else "-",
                "markdown_chars": md_path.stat().st_size if md_path.exists() else 0,
                "chunks": n,
            }
        )
        continue

    print(f"\n[{pdf_no}/{total_pdfs}] {pdf_path.name}")

    if md_path.exists():
        print(f"      → Markdown found — skipping Docling parse")
        print(f"      → Chunking from {md_path.name} ...")
        new_docs = chunk_from_markdown(md_path, pdf_path)
        md_chars = md_path.stat().st_size
    else:
        print("      → Running Docling (one PDF at a time, memory-safe)...")
        print_ram("before parse")
        result = run_docling_parse(pdf_path)
        dl_doc = result.document
        md = str(dl_doc.export_to_markdown())
        md_path.write_text(md, encoding="utf-8")
        md_chars = len(md)
        print(f"      → Saved markdown: {md_path.name} ({md_chars:,} characters)")
        print("      → Chunking with HybridChunker...")
        new_docs = chunk_with_hybrid(dl_doc, pdf_path)
        del result, dl_doc
        free_memory()
        print_ram("after parse + cleanup")

    documents.extend(new_docs)
    chunked_sources.add(pdf_path.name)
    save_all_chunks(documents)

    parse_report.append(
        {
            "pdf": pdf_path.name,
            "status": "done",
            "markdown_file": md_path.name,
            "markdown_chars": md_chars,
            "chunks": len(new_docs),
        }
    )
    print(f"      → Chunks for this PDF: {len(new_docs)}")
    print(f"      → Updated {CHUNKS_PATH.name} ({len(documents)} total chunks)")
    print(f"      ✓ Done: {pdf_path.name}")
    free_memory()

print("\n" + "=" * 60)
print("PARSE SUMMARY")
print("=" * 60)
for row in parse_report:
    print(
        f"  • {row['pdf']} [{row['status']}]\n"
        f"      markdown: {row['markdown_file']} ({row['markdown_chars']:,} chars)\n"
        f"      chunks:   {row['chunks']}"
    )
print(f"\nTotal chunks: {len(documents)}")
print(f"Saved: {CHUNKS_PATH}")

if documents:
    sample = documents[-1]
    print("\n--- Sample chunk (last added, first 400 chars) ---")
    print(f"Source: {sample.metadata.get('source')} | Page: {sample.metadata.get('page_number')}")
    print(sample.page_content[:400])

PARSING & CHUNKING
Chunks file: /home/abiram/Prayoga/Marga/RAG With LangChain/rag_index/chunks.json

Resume check:
  • AI Engineering.pdf: markdown=yes | chunks=yes
  • Data Science.pdf: markdown=yes | chunks=yes
  • ML.pdf: markdown=yes | chunks=yes
  • Python.pdf: markdown=no | chunks=no

[1/4] SKIP AI Engineering.pdf — already in chunks.json (1305 chunks)

[2/4] SKIP Data Science.pdf — already in chunks.json (454 chunks)

[3/4] SKIP ML.pdf — already in chunks.json (165 chunks)

[4/4] Python.pdf
      → Running Docling (one PDF at a time, memory-safe)...
      → RAM: 1008 MB (before parse)
      → 68 pages — layout model runs on each (no live progress bar)


Loading weights: 100%|██████████| 770/770 [00:00<00:00, 900.20it/s] 


      → Docling finished in 0m 28s
      → Saved markdown: Python.md (50,595 characters)
      → Chunking with HybridChunker...
      → RAM: 2444 MB (after parse + cleanup)
      → Chunks for this PDF: 116
      → Updated chunks.json (2040 total chunks)
      ✓ Done: Python.pdf

PARSE SUMMARY
  • AI Engineering.pdf [skipped]
      markdown: AI Engineering.md (1,226,770 chars)
      chunks:   1305
  • Data Science.pdf [skipped]
      markdown: Data Science.md (434,029 chars)
      chunks:   454
  • ML.pdf [skipped]
      markdown: ML.md (153,849 chars)
      chunks:   165
  • Python.pdf [done]
      markdown: Python.md (50,595 chars)
      chunks:   116

Total chunks: 2040
Saved: /home/abiram/Prayoga/Marga/RAG With LangChain/rag_index/chunks.json

--- Sample chunk (last added, first 400 chars) ---
Source: Python.pdf | Page: 68
Explore More www.tutort.net Watch us on Youtube - Read more on Quora Explore our courses Data Science & Machine Learning Follow us on Full Stack Data Science (AI 

## Step 4 — Build embeddings and FAISS index

Uses **`BAAI/bge-base-en-v1.5`** (downloads once from Hugging Face).

In [4]:
if not documents:
    raise ValueError("No chunks to embed. Check parsing step — PDFs may be empty or OCR failed.")

print("=" * 60)
print("EMBEDDINGS & FAISS INDEX")
print("=" * 60)
print(f"Loading embedding model: {EMBEDDING_MODEL} ...")

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    encode_kwargs={"normalize_embeddings": True},
)

texts = [d.page_content for d in documents]
print(f"Embedding {len(texts)} chunks (this may take a minute)...")
vectors = np.asarray(embeddings.embed_documents(texts), dtype=np.float32)
faiss.normalize_L2(vectors)

index = faiss.IndexFlatIP(vectors.shape[1])
index.add(vectors)
faiss.write_index(index, str(INDEX_PATH))

metadata = [d.metadata | {"text": d.page_content} for d in documents]
METADATA_PATH.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

print(f"\n✓ FAISS index:  {INDEX_PATH}  ({index.ntotal} vectors, dim={vectors.shape[1]})")
print(f"✓ Metadata:     {METADATA_PATH}")
print(f"✓ Chunks JSON:  {CHUNKS_PATH}")
print(f"✓ Markdown dir: {PARSED_MD}")
print("\nIndex build complete. Streamlit can use rag_index/ now.")

EMBEDDINGS & FAISS INDEX
Loading embedding model: BAAI/bge-base-en-v1.5 ...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1028.07it/s]


Embedding 2040 chunks (this may take a minute)...

✓ FAISS index:  /home/abiram/Prayoga/Marga/RAG With LangChain/rag_index/faiss.index  (2040 vectors, dim=768)
✓ Metadata:     /home/abiram/Prayoga/Marga/RAG With LangChain/rag_index/metadata.json
✓ Chunks JSON:  /home/abiram/Prayoga/Marga/RAG With LangChain/rag_index/chunks.json
✓ Markdown dir: /home/abiram/Prayoga/Marga/RAG With LangChain/rag_index/parsed_markdown

Index build complete. Streamlit can use rag_index/ now.


## Step 5 — Test retrieval

Runs a sample query against the FAISS index and prints the top matches.

In [7]:
def search(query: str, k: int = 5):
    qv = np.asarray([embeddings.embed_query(query)], dtype=np.float32)
    faiss.normalize_L2(qv)
    return index.search(qv, k)

query = "A speech recognition system has 75% of training errors attributed to background noise. How would you design an error analysis process to identify the most impactful improvements while balancing practical constraints?"
scores, indices = search(query)

print("=" * 60)
print("RETRIEVAL TEST")
print("=" * 60)
print(f"Query: {query}\n")

for rank, idx in enumerate(indices[0], start=1):
    if idx == -1:
        continue
    item = metadata[idx]
    score = float(scores[0][rank - 1])
    print(f"--- Rank {rank} | score {score:.4f} ---")
    print(f"Source: {item.get('source')}  |  Page: {item.get('page_number')}")
    print(item["text"][:450])
    print()

RETRIEVAL TEST
Query: A speech recognition system has 75% of training errors attributed to background noise. How would you design an error analysis process to identify the most impactful improvements while balancing practical constraints?

--- Rank 1 | score 0.8181 ---
Source: ML.pdf  |  Page: None
Other problems are harder. For example, suppose that you are building a speech recognition system, and find that 14% of the audio clips have so much background noise or are so unintelligible that even a human cannot recognize what was said. In this case, even the most 'optimal' speech recognition system might have error around 14%. Suppose that on this speech recognition problem, your algorithm achieves: - Training error = 15% - Dev error = 30% T

--- Rank 2 | score 0.7727 ---
Source: ML.pdf  |  Page: None
I recommend that you: (i) Try to understand what properties of the data differ between the training and the dev set distributions. (ii) Try to find more training data that better matches t

## Step 6 — Test answer with Qwen (Ollama)

Requires Ollama running with `qwen3:8b` pulled:
```bash
ollama pull qwen3:8b
```

In [8]:
question = "What does the hyperparameter 'k' represent in the k-nearest neighbors algorithm?"
scores, indices = search(question)

context_parts = []
for rank, idx in enumerate(indices[0], start=1):
    if idx == -1:
        continue
    item = metadata[idx]
    context_parts.append(
        f"[{rank}] {item['source']} p.{item.get('page_number')}: {item['text']}"
    )
context = "\n\n".join(context_parts)

print("=" * 60)
print("GENERATION TEST")
print("=" * 60)
print(f"Question: {question}")
print(f"Retrieved {len(context_parts)} chunks. Calling {OLLAMA_MODEL}...\n")

prompt = f"""Answer using only the context below. If the answer is not in the context, say you do not know.

Context:
{context}

Question: {question}

Answer:"""

response = chat(
    model=OLLAMA_MODEL,
    messages=[{"role": "user", "content": prompt}],
    think=False,
)

print("--- Answer ---")
print(response.message.content)

GENERATION TEST
Question: What does the hyperparameter 'k' represent in the k-nearest neighbors algorithm?
Retrieved 5 chunks. Calling qwen3:8b...

--- Answer ---
The hyperparameter 'k' in the k-nearest neighbors algorithm represents the number of nearest neighbors to consider when making a prediction. It determines how many instances from the training dataset are used to classify a new, unseen instance. A higher value of 'k' leads to a smoother decision boundary, while a lower value makes the model more sensitive to the local structure of the data.
